In [ ]:
pip install ucimlrepo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import recall_score, precision_score, f1_score

# ✅ MISSING IMPORT (this was the main error)
from ucimlrepo import fetch_ucirepo


# ==========================================
# 1. Load Dataset
# ==========================================
heart_disease = fetch_ucirepo(id=45)

X_raw = heart_disease.data.features
y_raw = heart_disease.data.targets

print(heart_disease.metadata)
print(heart_disease.variables)

# ==========================================
# 2. Combine Dataset
# ==========================================
dataset = pd.concat([X_raw, y_raw], axis=1)


dataset = dataset.replace('?', np.nan)

# Convert columns properly
dataset['ca'] = pd.to_numeric(dataset['ca'])
dataset['thal'] = pd.to_numeric(dataset['thal'])

# Fill missing values
dataset['ca'] = dataset['ca'].fillna(dataset['ca'].median())
dataset['thal'] = dataset['thal'].fillna(dataset['thal'].mode()[0])

# Now drop remaining NaNs
dataset = dataset.dropna()

# Convert target
dataset['num'] = np.where(dataset['num'] == 0, -1, 1)

# Features & Labels
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

print(dataset.head())

{'uci_id': 45, 'name': 'Heart Disease', 'repository_url': 'https://archive.ics.uci.edu/dataset/45/heart+disease', 'data_url': 'https://archive.ics.uci.edu/static/public/45/data.csv', 'abstract': '4 databases: Cleveland, Hungary, Switzerland, and the VA Long Beach', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 303, 'num_features': 13, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': ['Age', 'Sex'], 'target_col': ['num'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1989, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C52P4X', 'creators': ['Andras Janosi', 'William Steinbrunn', 'Matthias Pfisterer', 'Robert Detrano'], 'intro_paper': {'ID': 231, 'type': 'NATIVE', 'title': 'International application of a new probability algorithm for the diagnosis of coronary artery disease.', 'authors': 'R. Detrano, A. Jánosi, W. Steinbrunn, M

In [ ]:
# ==========================================
# 3. Train-Test Split
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

# ==========================================
# 4. Min-Max Scaling
# ==========================================
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================================
# 5. Train SVM
# ==========================================
clf = svm.SVC(
    kernel='rbf',
    gamma=1 / X_train.shape[1],
    C=1.0
)
clf.fit(X_train, y_train)

print("Training completed")
print("Number of support vectors:", clf.support_vectors_.shape[0])

# ==========================================
# 6. Single Sample Test
# ==========================================
sample_index = 0

x_single = X_test[sample_index]
y_single = y_test[sample_index]

decision_score = clf.decision_function([x_single])

print("Ground truth:", y_single)
print("Decision score:", decision_score[0])

# ==========================================
# 7. Save Files
# ==========================================
np.savetxt("support_vectors_rbf.txt", clf.support_vectors_)
np.savetxt("dual_coeff_rbf.txt", clf.dual_coef_)
np.savetxt("intercept_rbf.txt", clf.intercept_)
np.savetxt("xtest_rbf.txt", x_single)
np.savetxt("ytest_rbf.txt", [y_single])
np.savetxt("yclassificationscore.txt", decision_score)

print("All files saved successfully.")

Training completed
Number of support vectors: 128
Ground truth: -1
Decision score: -0.7934602960241713
All files saved successfully.


In [ ]:

# ==========================================
# 8. Print Test Data
# ==========================================
print("test datasets")
for i in range(len(X_test)):
    print(X_test[i], "\t", y_test[i], "\n")

# ==========================================
# 9. DataFrame View
# ==========================================
x_test_df = pd.DataFrame(X_test)
x_test_df['target'] = y_test

print("Test dataset:")
print(x_test_df)

test datasets
[0.42222222 1.         1.         0.3255814  0.21917808 0.
 1.         0.8778626  0.         0.         0.         0.
 0.        ] 	 -1 

[0.66666667 1.         1.         0.81395349 0.11415525 1.
 1.         0.14503817 0.         0.17857143 0.5        0.66666667
 0.75      ] 	 1 

[0.48888889 0.         0.66666667 0.41860465 0.29680365 0.
 1.         0.59541985 0.         0.08928571 0.         0.
 0.        ] 	 -1 

[0.88888889 0.         0.         0.53488372 0.25799087 0.
 0.         0.61068702 0.         0.32142857 0.         0.66666667
 0.        ] 	 -1 

[0.55555556 0.         0.66666667 0.76744186 0.17123288 0.
 0.         0.70229008 0.         0.         0.         0.33333333
 0.        ] 	 -1 

[0.33333333 1.         1.         0.30232558 0.09817352 0.
 0.         0.55725191 1.         0.5        1.         0.
 0.75      ] 	 1 

[0.28888889 1.         1.         0.53488372 0.2283105  0.
 0.         0.81679389 0.         0.         0.         0.
 0.        ] 	 -1 

In [ ]:
# ==========================================
# 10. Evaluation
# ==========================================
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))
print("accuracy:", accuracy_score(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

              precision    recall  f1-score   support

          -1       0.82      0.89      0.86        37
           1       0.81      0.71      0.76        24

    accuracy                           0.82        61
   macro avg       0.82      0.80      0.81        61
weighted avg       0.82      0.82      0.82        61

accuracy: 0.819672131147541
Precision: 0.8095238095238095
Recall: 0.7083333333333334
F1 Score: 0.7555555555555555
